# Analisis Telemetri Lintas Domain & Eksplorasi Data (EDA)
**Dataset:** `data/telemetry_dataset.csv` (4.500 sampel, 40 kolom, 8 skenario eksperimen)  
**Lingkup:** 5G O-RAN Multi-Slice (Radio gNB, Core UPF, & 3GPP NWDAF AnLF)


## Tahap 1: Pemuatan Data & Demarkasi Kolom (Data Ingestion & Demarcation)
- **Tujuan:** Memuat dataset telemetri multi-domain dan memetakan taksonomi kolom sesuai peran fungsional arsitektur sistem.
- **Target:** Dataset 4.500 sampel termuat tanpa error dan 40 kolom terkelompokkan ke dalam 5 grup taksonomi (Metadata, SLA Probe, RAN, Core UPF, dan 3GPP NWDAF).


In [ ]:
# 1. Import library analisis dan konfigurasi visualisasi
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['figure.dpi'] = 120

DATASET_PATH = os.path.join('..', 'data', 'telemetry_dataset.csv')
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.join('data', 'telemetry_dataset.csv')
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.join('data', 'telemetry_dataset_10min_complete.csv')

df = pd.read_csv(DATASET_PATH)
print(f'Dataset berhasil dimuat dari: {os.path.abspath(DATASET_PATH)}')
print(f'Dimensi Dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')


In [ ]:
# 2. Pengelompokan taksonomi kolom resmi sesuai Matriks Kapabilitas
METADATA_COLS = [
    'timestamp_utc', 'elapsed_sec', 'run_id', 'scrape_ok', 'scrape_latency_ms',
    'event_phase_id', 'event_scenario', 'event_domain', 'event_component',
    'rapp_diagnosed_bottleneck', 'rapp_confidence'
]

SLA_PROBE_COLS = [
    'e2e_slice1_tput_mbps', 'e2e_slice1_jitter_ms', 'e2e_slice1_rtt_ms', 'e2e_slice1_loss_pct',
    'e2e_slice2_rtt_ms', 'e2e_slice2_loss_pct', 'e2e_probe_freshness'
]

INFRA_RAN_COLS = [
    'ran_prb_cell1', 'ran_prb_cell2', 'ran_mcs_cell1', 'ran_mcs_cell2',
    'ran_bler_cell1', 'ran_bler_cell2', 'ran_snr_cell1', 'ran_snr_cell2'
]

INFRA_CORE_COLS = [
    'estimated_load_embb', 'estimated_load_urllc',
    'upf_backlog_bytes_embb', 'upf_backlog_bytes_urllc',
    'upf_drops_embb', 'upf_drops_urllc'
]

NWDAF_COLS = [
    'nwdaf_load_embb', 'nwdaf_load_urllc', 'nwdaf_comm_ul_bytes', 'nwdaf_comm_dl_bytes',
    'nwdaf_comm_duration_sec', 'nwdaf_comm_vol_std_bytes', 'nwdaf_comm_freshness', 'nwdaf_api_status'
]

classified_total = len(METADATA_COLS) + len(SLA_PROBE_COLS) + len(INFRA_RAN_COLS) + len(INFRA_CORE_COLS) + len(NWDAF_COLS)
print(f'Total Kolom Terklasifikasi: {classified_total} / {df.shape[1]} kolom')


## Tahap 2: Pembersihan Data & Audit Kualitas (Data Cleansing & Quality Audit)
- **Tujuan:** Mengaudit kelengkapan data numerik, mendeteksi nilai hilang (`NaN`), mengidentifikasi fitur konstan (varians nol), dan menghitung distribusi kuantitatif SLA (persentil p50, p95, p99 serta rasio pelanggaran).
- **Target:** Tabel audit kualitas 40 kolom, ringkasan integritas 8 run eksperimen dengan jeda waktu nyata antar-run (diselaraskan UTC & WIB), dan tabel persentil SLA layanan.


In [ ]:
# 1. Audit kelengkapan dan variansi per kolom
audit_rows = []
for col in df.columns:
    total_count = len(df[col])
    nan_count = int(df[col].isnull().sum())
    valid_count = total_count - nan_count
    pct_valid = round((valid_count / total_count) * 100.0, 1)
    
    is_numeric = pd.api.types.is_numeric_dtype(df[col])
    num_unique = df[col].nunique(dropna=True)
    is_constant = (num_unique <= 1)
    
    val_min = round(float(df[col].min()), 2) if is_numeric and valid_count > 0 else 'N/A'
    val_max = round(float(df[col].max()), 2) if is_numeric and valid_count > 0 else 'N/A'
    
    audit_rows.append({
        'Kolom': col,
        'Tipe': 'Numeric' if is_numeric else 'String',
        'Valid': valid_count,
        'NaN': nan_count,
        '% Valid': f'{pct_valid}%',
        'Unique': num_unique,
        'Status Variasi': '100% NaN' if valid_count == 0 else ('Konstan' if is_constant else 'Bervariasi'),
        'Min': str(val_min),
        'Max': str(val_max)
    })

df_audit = pd.DataFrame(audit_rows)
pd.set_option('display.max_rows', 45)
display(df_audit)

cols_all_nan = df_audit[df_audit['Status Variasi'] == '100% NaN']['Kolom'].tolist()
cols_constant = df_audit[df_audit['Status Variasi'] == 'Konstan']['Kolom'].tolist()

print(f'Kolom 100% NaN ({len(cols_all_nan)}): {cols_all_nan}')
print(f'Kolom Konstan / Varians Nol ({len(cols_constant)}): {cols_constant}')


In [ ]:
# 2. Rincian integritas scraping dan jeda waktu fisik antar-run eksperimen (UTC & WIB)
run_summary = []
prev_end_time = None

for run_id, grp in df.groupby('run_id', sort=False):
    scen = grp['event_scenario'].iloc[0]
    dom = grp['event_domain'].iloc[0]
    n_rows = len(grp)
    scrape_ok_pct = round((grp['scrape_ok'].sum() / n_rows) * 100.0, 1)
    
    start_dt = pd.to_datetime(grp['timestamp_utc'].iloc[0])
    end_dt = pd.to_datetime(grp['timestamp_utc'].iloc[-1])
    run_dur = round((end_dt - start_dt).total_seconds(), 1)
    
    start_utc_str = start_dt.strftime('%H:%M:%S UTC')
    start_wib_str = (start_dt + pd.Timedelta(hours=7)).strftime('%H:%M:%S WIB')
    
    if prev_end_time is not None:
        gap_sec = round((start_dt - prev_end_time).total_seconds(), 1)
        gap_str = f'{int(gap_sec // 60)}m {int(gap_sec % 60)}s ({gap_sec}s)'
    else:
        gap_str = 'Start Run'
    prev_end_time = end_dt
    
    prb_c1_mean = round(float(grp['ran_prb_cell1'].mean()), 1)
    backlog_max = int(grp['upf_backlog_bytes_embb'].max())
    nwdaf_status = int(grp['nwdaf_api_status'].mean())
    nwdaf_valid_rows = int(grp['nwdaf_comm_ul_bytes'].notnull().sum())
    
    run_summary.append({
        'Run ID': run_id,
        'Skenario': scen,
        'Domain': dom,
        'Waktu Mulai': f'{start_utc_str} ({start_wib_str})',
        'Sampel': n_rows,
        'Durasi (s)': run_dur,
        'Jeda Pra-Run': gap_str,
        'Scrape OK': f'{scrape_ok_pct}%',
        'PRB C1 Avg (%)': f'{prb_c1_mean}%',
        'Backlog Max (B)': f'{backlog_max:,}',
        'NWDAF Status': nwdaf_status,
        'NWDAF Valid': f'{nwdaf_valid_rows}/{n_rows}'
    })

df_run_summary = pd.DataFrame(run_summary)
display(df_run_summary)


In [ ]:
# 3. Distribusi kuantitatif metrik SLA: Persentil RTT, Loss, dan Rasio Pelanggaran Ambang
sla_rows = []
for run_id, grp in df.groupby('run_id', sort=False):
    scen = grp['event_scenario'].iloc[0]
    n_total = len(grp)
    
    s1_rtt = grp['e2e_slice1_rtt_ms']
    s1_nan_count = int(s1_rtt.isnull().sum())
    s1_outage_pct = round((s1_nan_count / n_total) * 100.0, 1)
    s1_p50 = round(float(s1_rtt.quantile(0.50)), 2) if (n_total - s1_nan_count) > 0 else 'N/A'
    s1_p95 = round(float(s1_rtt.quantile(0.95)), 2) if (n_total - s1_nan_count) > 0 else 'N/A'
    s1_p99 = round(float(s1_rtt.quantile(0.99)), 2) if (n_total - s1_nan_count) > 0 else 'N/A'
    s1_loss_mean = round(float(grp['e2e_slice1_loss_pct'].mean()), 1)
    
    s2_rtt = grp['e2e_slice2_rtt_ms']
    s2_p50 = round(float(s2_rtt.quantile(0.50)), 2)
    s2_p95 = round(float(s2_rtt.quantile(0.95)), 2)
    s2_p99 = round(float(s2_rtt.quantile(0.99)), 2)
    s2_loss_mean = round(float(grp['e2e_slice2_loss_pct'].mean()), 1)
    s2_viol_count = int((s2_rtt > 20.0).sum())
    s2_viol_pct = round((s2_viol_count / n_total) * 100.0, 1)
    
    sla_rows.append({
        'Skenario': scen,
        'eMBB Outage (%)': f'{s1_outage_pct}%',
        'eMBB Loss (%)': f'{s1_loss_mean}%',
        'eMBB RTT p50 (ms)': s1_p50,
        'eMBB RTT p95 (ms)': s1_p95,
        'eMBB RTT p99 (ms)': s1_p99,
        'URLLC Loss (%)': f'{s2_loss_mean}%',
        'URLLC RTT p50 (ms)': s2_p50,
        'URLLC RTT p95 (ms)': s2_p95,
        'URLLC RTT p99 (ms)': s2_p99,
        'URLLC >20ms Viol (%)': f'{s2_viol_pct}%'
    })

df_sla_dist = pd.DataFrame(sla_rows)
display(df_sla_dist)


## Tahap 3: Pra-pemrosesan Data & Rekayasa Fitur (Data Preprocessing & Feature Engineering)
- **Tujuan:** Menghitung fitur turunan fisik, memetakan distribusi kelas target, dan mendemarkasi set fitur masukan secara ketat ke dalam **4 protokol studi ablasi simetris** (Set A: RAN-Only, Set B: NWDAF-Only, Set C: Cross-Domain Fused - Utama, Set D: Kernel Reference - Upper Bound).
- **Penyelarasan Metodologis:** Kolom `nwdaf_comm_duration_sec` **dieksklusi** dari kandidat model utama untuk mencegah model belajar umur kumulatif sesi PDU atau urutan kronologis eksperimen (*session-age chronological leakage*). Kolom ini tetap dipertahankan pada level dataset untuk audit dan visualisasi EDA.
- **Target:** Variabel turunan terhitung (`upf_backlog_kb`), distribusi frekuensi kelas target `event_domain`, dan penetapan resmi 4 set fitur masukan untuk evaluasi komparatif.


In [ ]:
# 1. Perhitungan fitur turunan
df['upf_backlog_kb'] = df['upf_backlog_bytes_embb'] / 1024.0
df['ran_prb_pct'] = df['ran_prb_cell1'].clip(0.0, 100.0)

# 2. Distribusi frekuensi kelas target domain
target_dist = df['event_domain'].value_counts()
target_pct = df['event_domain'].value_counts(normalize=True) * 100.0
df_class_dist = pd.DataFrame({
    'Jumlah Sampel': target_dist,
    'Proporsi (%)': target_pct.round(2)
})
print('=== DISTRIBUSI KELAS TARGET (event_domain) ===')
display(df_class_dist)

# 3. Demarkasi Empat Set Fitur Masukan Resmi (Protokol Studi Ablasi Simetris Lintas Domain)
FEATURE_SET_A_RAN = [
    'ran_prb_cell1', 'ran_snr_cell1'
]

FEATURE_SET_B_NWDAF_ONLY = [
    'nwdaf_comm_ul_bytes', 'nwdaf_comm_vol_std_bytes'
]

FEATURE_SET_C_FUSED = [
    'ran_prb_cell1', 'ran_snr_cell1',
    'nwdaf_comm_ul_bytes', 'nwdaf_comm_vol_std_bytes'
]

FEATURE_SET_D_KERNEL_REF = [
    'ran_prb_cell1', 'ran_snr_cell1',
    'upf_backlog_bytes_embb', 'upf_drops_embb', 'estimated_load_embb'
]

print(f'1. Set A (RAN-Only Baseline): {len(FEATURE_SET_A_RAN)} fitur ->', FEATURE_SET_A_RAN)
print(f'2. Set B (NWDAF-Only Core Baseline): {len(FEATURE_SET_B_NWDAF_ONLY)} fitur ->', FEATURE_SET_B_NWDAF_ONLY)
print(f'3. Set C (Cross-Domain Fused - Eksperimen Utama): {len(FEATURE_SET_C_FUSED)} fitur ->', FEATURE_SET_C_FUSED)
print(f'4. Set D (RAN + Kernel Host OAM - Referensi Pembanding): {len(FEATURE_SET_D_KERNEL_REF)} fitur ->', FEATURE_SET_D_KERNEL_REF)


## Tahap 4: Analisis Data Eksploratif (Exploratory Data Analysis / EDA)
- **Tujuan:** Menampilkan visualisasi runtun waktu lintas run dengan garis terputus pada jeda nyata, menganalisis dinamika tangga telemetri 3GPP NWDAF AnLF, mengevaluasi kondisi pasangan terkendali (S2 vs S6 serta perbandingan kondisi antar-run S9 -> S10_A -> S10_B), menghitung korelasi internal/global, dan secara khusus **menganalisis korelasi metrik terhadap kondisi jaringan (BASE, RAN, CORE, JOINT)**.
- **Target:** Plot runtun waktu multi-panel bersekat jeda fisik, plot tangga berkala NWDAF, boxplot pasangan terkendali, heatmap korelasi Spearman teranotasi ($N$), serta matriks korelasi dan barplot Correlation Ratio (Eta) terhadap kondisi domain target.


In [ ]:
# 1. Dinamika runtun waktu multi-panel (Garis terputus pada jeda fisik antar-run)
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)

current_idx = 0
prev_end_dt = None

for run_id, grp in df.groupby('run_id', sort=False):
    scen = grp['event_scenario'].iloc[0]
    start_dt = pd.to_datetime(grp['timestamp_utc'].iloc[0])
    end_dt = pd.to_datetime(grp['timestamp_utc'].iloc[-1])
    n_rows = len(grp)
    x_run = np.arange(current_idx, current_idx + n_rows)
    
    # Throughput laju pengiriman UE-1
    axes[0].plot(x_run, grp['e2e_slice1_tput_mbps'], color='royalblue', lw=1.2)
    
    # Proxy Alokasi Grant PUSCH Cell-1
    axes[1].plot(x_run, grp['ran_prb_cell1'], color='darkorange', lw=1.2)
    
    # Kedalaman Antrean UPF (Backlog KB)
    axes[2].plot(x_run, grp['upf_backlog_kb'], color='crimson', lw=1.2)
    
    # Latensi Canary RTT UE-1 & UE-2
    axes[3].plot(x_run, grp['e2e_slice1_rtt_ms'], color='blue', lw=1.0, alpha=0.8)
    axes[3].plot(x_run, grp['e2e_slice2_rtt_ms'], color='green', lw=1.0)
    
    # Garis sekat antar-run
    for ax in axes:
        ax.axvline(current_idx, color='black', ls='-', lw=1.2, alpha=0.7)
    
    # Label skenario dan jeda waktu
    mid_point = current_idx + n_rows / 2
    label_clean = scen.replace('_', '\n')
    axes[0].text(mid_point, 86, label_clean, ha='center', va='top', fontsize=7.5, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='gray'))
    
    if prev_end_dt is not None:
        gap_sec = round((start_dt - prev_end_dt).total_seconds(), 0)
        gap_label = f'Jeda: {int(gap_sec // 60)}m {int(gap_sec % 60)}s'
        axes[1].text(current_idx, 105, gap_label, ha='center', va='bottom', fontsize=7, color='maroon', rotation=90)
    
    prev_end_dt = end_dt
    current_idx += n_rows

# Konfigurasi Sumbu & Label
axes[0].set_ylabel('Laju Kirim UE-1\n(Offered Mbps)')
axes[0].set_ylim(-2, 95)
axes[0].grid(True, ls='--', alpha=0.5)

axes[1].axhline(100.0, color='red', ls=':', alpha=0.7, label='Batas Saturasi (100% Grant)')
axes[1].set_ylabel('Grant Proxy (%)')
axes[1].set_ylim(-5, 120)
axes[1].grid(True, ls='--', alpha=0.5)
axes[1].legend(loc='upper right')

axes[2].axhline(300.0, color='purple', ls=':', alpha=0.7, label='Batas Buffer TBF (~300 KB)')
axes[2].set_ylabel('Backlog (KB)')
axes[2].set_ylim(-10, 360)
axes[2].grid(True, ls='--', alpha=0.5)
axes[2].legend(loc='upper right')

axes[3].axhline(20.0, color='darkgreen', ls='--', alpha=0.6, label='Ambang SLA URLLC (20 ms)')
axes[3].set_ylabel('RTT (ms)')
axes[3].set_yscale('log')
axes[3].set_ylim(8, 1500)
axes[3].grid(True, ls='--', alpha=0.5)
axes[3].legend(loc='upper right')

axes[3].set_xlabel('Indeks Sampel Sekuensial (Garis terputus pada sekat hitam menandai jeda fisik antar-run)')
plt.suptitle('Dinamika Telemetri Lintas Domain di 8 Skenario Eksperimen (Batas Run Terputus & Teranotasi)', y=0.99, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 2. Dinamika tangga telemetri 3GPP NWDAF AnLF (Pembaruan nilai sumber ~30 detik; polling collector 2 detik)
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

current_idx = 0
for run_id, grp in df.groupby('run_id', sort=False):
    scen = grp['event_scenario'].iloc[0]
    n_rows = len(grp)
    x_run = np.arange(current_idx, current_idx + n_rows)
    
    # Volume Uplink (MB) via step plot
    axes[0].plot(x_run, grp['nwdaf_comm_ul_bytes'] / 1e6, color='teal', lw=1.3, drawstyle='steps-post')
    
    # Standar Deviasi Volume (KB) via step plot
    axes[1].plot(x_run, grp['nwdaf_comm_vol_std_bytes'] / 1e3, color='darkmagenta', lw=1.3, drawstyle='steps-post')
    
    for ax in axes:
        ax.axvline(current_idx, color='black', ls='-', lw=1.0, alpha=0.6)
    
    mid_point = current_idx + n_rows / 2
    label_clean = scen.replace('_', '\n')
    axes[0].text(mid_point, axes[0].get_ylim()[1] * 0.9 if current_idx > 0 else 6000, label_clean,
                 ha='center', va='top', fontsize=7.5, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='gray'))
    current_idx += n_rows

axes[0].set_ylabel('NWDAF UL Vol (MB)')
axes[0].grid(True, ls='--', alpha=0.5)
axes[0].set_title('Volume Uplink Kumulatif Tercatat pada NWDAF AnLF (Format Tangga / Steps-Post)', fontsize=10, fontweight='bold')

axes[1].set_ylabel('NWDAF Vol Std (KB)')
axes[1].grid(True, ls='--', alpha=0.5)
axes[1].set_title('Standar Deviasi Volume Antar-Record Sesi pada NWDAF AnLF', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Indeks Sampel Sekuensial')

plt.suptitle('Karakteristik Tangga Telemetri 3GPP NWDAF AnLF (Pembaruan Nilai Sumber Sekitar 30 Detik; Polling Collector 2s)', y=1.01, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 3. Evaluasi pasangan terkendali: S2 vs S6 dan Perbandingan Kondisi Antar-Run Pemulihan Bertahap
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# A. Perbandingan RTT UE-1 pada S2 vs S6
df_s2_s6 = df[df['event_scenario'].isin(['S2_NORMAL_HIGH', 'S6_CORE_QUEUE_TBF'])].copy()
sns.boxplot(data=df_s2_s6, x='event_scenario', y='e2e_slice1_rtt_ms', hue='event_scenario', legend=False, ax=axes[0], palette=['lightblue', 'salmon'])
axes[0].set_title('Perbandingan RTT UE-1 (Beban 20M)', fontweight='bold')
axes[0].set_ylabel('RTT (ms)')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['S2 (Tanpa TC)', 'S6 (TBF 12M)'])
axes[0].grid(True, ls='--', alpha=0.5)

# B. Perbandingan UPF Backlog pada S2 vs S6
sns.boxplot(data=df_s2_s6, x='event_scenario', y='upf_backlog_bytes_embb', hue='event_scenario', legend=False, ax=axes[1], palette=['lightblue', 'salmon'])
axes[1].set_title('Perbandingan UPF Backlog (Beban 20M)', fontweight='bold')
axes[1].set_ylabel('Backlog Bytes')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['S2 (Tanpa TC)', 'S6 (TBF 12M)'])
axes[1].grid(True, ls='--', alpha=0.5)

# C. Perbandingan Kondisi Antar-Run Pemulihan Bertahap: S9 -> S10_A -> S10_B
df_rec = df[df['event_scenario'].isin(['S9_JOINT_OVERLOAD', 'S10_A_RELEASE_RAN', 'S10_B_RELEASE_CORE'])].copy()
rec_order = ['S9_JOINT_OVERLOAD', 'S10_A_RELEASE_RAN', 'S10_B_RELEASE_CORE']
sns.boxplot(data=df_rec, x='event_scenario', y='upf_backlog_bytes_embb', hue='event_scenario', legend=False, order=rec_order, ax=axes[2], palette=['purple', 'crimson', 'lightgreen'])
axes[2].set_title('Kondisi Antar-Run Pemulihan Bertahap', fontweight='bold')
axes[2].set_ylabel('Backlog Bytes')
axes[2].set_xticks([0, 1, 2])
axes[2].set_xticklabels(['S9 (Joint)', 'S10_A (Rilis RAN)', 'S10_B (Rilis Core)'], rotation=10)
axes[2].grid(True, ls='--', alpha=0.5)

plt.suptitle('Analisis Pasangan Terkendali (S2 vs S6) & Perbandingan Kondisi Antar-Run Pemulihan Bertahap', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 4. Matriks korelasi Spearman dan audit jumlah pasangan observasi berpasangan (N)
candidate_features = [
    'e2e_slice1_tput_mbps', 'e2e_slice1_loss_pct', 'e2e_slice1_rtt_ms',
    'e2e_slice2_rtt_ms', 'e2e_slice2_loss_pct',
    'ran_prb_cell1', 'ran_snr_cell1',
    'upf_backlog_bytes_embb', 'upf_drops_embb', 'estimated_load_embb',
    'nwdaf_comm_ul_bytes', 'nwdaf_comm_vol_std_bytes'
]

# Hitung matriks jumlah observasi berpasangan valid (N)
n_matrix = pd.DataFrame(index=candidate_features, columns=candidate_features)
for col1 in candidate_features:
    for col2 in candidate_features:
        n_matrix.loc[col1, col2] = int(df[[col1, col2]].dropna().shape[0])

print('=== JUMLAH PASANGAN OBSERVASI VALID (PAIRWISE COMPLETE N) ===')
display(n_matrix)

# Korelasi Spearman Global
corr_global = df[candidate_features].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr_global, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1.0, vmax=1.0, linewidths=0.5)
plt.title('Matriks Korelasi Spearman Global (Pasangan RTT eMBB: N=3.300 Radio/Kernel & N=3.292 NWDAF)', fontweight='bold')
plt.tight_layout()
plt.show()

# Korelasi Spearman Internal Skenario Core Bottleneck (S6)
df_s6 = df[df['event_scenario'] == 'S6_CORE_QUEUE_TBF'][candidate_features].dropna(axis=1, how='all')
varying_s6 = [c for c in df_s6.columns if df_s6[c].nunique() > 1]
corr_s6 = df_s6[varying_s6].corr(method='spearman')
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr_s6, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1.0, vmax=1.0, linewidths=0.5)
plt.title('Matriks Korelasi Spearman Internal Skenario S6 (Core TBF)', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 5. Pengecekan Korelasi Metrik Telemetri terhadap Kondisi Jaringan (BASE, RAN, CORE, JOINT)
dummies = pd.get_dummies(df['event_domain'], prefix='cond', dtype=float)
df_corr_cond = pd.concat([df[candidate_features], dummies], axis=1)

# Matriks Korelasi Spearman Metrik vs Kondisi (Domain)
corr_cond = df_corr_cond.corr(method='spearman')
cond_cols = ['cond_BASELINE', 'cond_RAN', 'cond_CORE', 'cond_JOINT']
cond_display_names = ['BASELINE', 'RAN', 'CORE', 'JOINT']

metric_cond_corr = corr_cond.loc[candidate_features, cond_cols].copy()
metric_cond_corr.columns = cond_display_names

print('=== MATRIKS KORELASI SPEARMAN METRIK TERHADAP KONDISI JARINGAN ===')
display(metric_cond_corr.round(3))

# Correlation Ratio (Eta) untuk mengukur kekuatan asosiasi non-linear keseluruhan
def correlation_ratio(categories, measurements):
    fcat, _ = pd.factorize(categories)
    cat_num = np.max(fcat) + 1
    y_avg_total = np.mean(measurements)
    cat_means = np.zeros(cat_num)
    cat_counts = np.zeros(cat_num)
    for i in range(cat_num):
        sub = measurements[fcat == i]
        cat_means[i] = np.mean(sub)
        cat_counts[i] = len(sub)
    n_sum = np.sum(cat_counts * ((cat_means - y_avg_total) ** 2))
    t_sum = np.sum((measurements - y_avg_total) ** 2)
    return np.sqrt(n_sum / t_sum) if t_sum != 0 else 0

eta_scores = []
for m in candidate_features:
    valid = df[[m, 'event_domain']].dropna()
    eta_val = correlation_ratio(valid['event_domain'], valid[m])
    eta_scores.append({
        'Fitur Telemetri': m,
        'Correlation Ratio (Eta)': round(eta_val, 3),
        'Eta Squared (Eta^2)': round(eta_val**2, 3),
        'Kekuatan Asosiasi': 'Sangat Kuat (>0.8)' if eta_val > 0.8 else ('Kuat (0.5-0.8)' if eta_val > 0.5 else ('Moderat (0.3-0.5)' if eta_val > 0.3 else 'Lemah (<0.3)'))
    })

df_eta = pd.DataFrame(eta_scores).sort_values(by='Correlation Ratio (Eta)', ascending=False)
print('=== CORRELATION RATIO (ETA) FITUR TERHADAP KONDISI JARINGAN ===')
display(df_eta)

# Visualisasi Heatmap Korelasi & Barplot Eta
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap Korelasi Biserial/Spearman (dengan anotasi khusus untuk NaN pada outage RTT eMBB)
sns.heatmap(metric_cond_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1.0, vmax=1.0,
            linewidths=0.5, ax=axes[0], cbar_kws={'label': 'Koefisien Korelasi Spearman'})
axes[0].set_title('Korelasi Spearman Metrik Telemetri terhadap Kondisi Domain', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Kondisi Jaringan (event_domain)', fontweight='bold')
axes[0].set_ylabel('Metrik Telemetri', fontweight='bold')

# Barplot Correlation Ratio (Eta)
y_pos = np.arange(len(df_eta))
axes[1].barh(y_pos, df_eta['Correlation Ratio (Eta)'], color='steelblue', alpha=0.85)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(df_eta['Fitur Telemetri'], fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel('Correlation Ratio (Eta) terhadap Kondisi Domain', fontweight='bold')
axes[1].set_title('Kekuatan Asosiasi Fitur Telemetri terhadap Kondisi Jaringan (Eta)', fontweight='bold', fontsize=11)
axes[1].set_xlim(0, 1.10)
axes[1].grid(True, ls='--', alpha=0.5)

for i, v in enumerate(df_eta['Correlation Ratio (Eta)']):
    axes[1].text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=8.5, fontweight='bold')

plt.suptitle('Analisis Korelasi & Asosiasi Fitur Telemetri terhadap Kondisi Jaringan (BASE, RAN, CORE, JOINT)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## Tahap 5: Evaluasi Distribusi & Keterpisahan Data per Domain (Feature Separability Assessment)
- **Tujuan:** Menampilkan visualisasi distribusi dan matriks statistik deskriptif fitur fisik pada masing-masing label domain target.
- **Target:** Boxplot komparatif dan tabel agregasi statistik deskriptif kuantitatif (`count`, `mean`, `std`, `median`, `min`, `max`) per domain target.


In [ ]:
# 1. Visualisasi distribusi fitur fisik per label domain
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
domain_order = ['BASELINE', 'RAN', 'CORE', 'JOINT']
palette = {'BASELINE': 'green', 'RAN': 'orange', 'CORE': 'red', 'JOINT': 'purple'}

# Distribusi PRB C1 per Domain
sns.boxplot(data=df, x='event_domain', y='ran_prb_cell1', hue='event_domain', legend=False, order=domain_order, ax=axes[0], palette=palette)
axes[0].set_title('Distribusi Proxy Alokasi Grant PUSCH Cell-1', fontweight='bold')
axes[0].set_xlabel('Domain Label')
axes[0].set_ylabel('Grant Proxy (%)')
axes[0].grid(True, ls='--', alpha=0.5)

# Distribusi UPF Backlog per Domain
sns.boxplot(data=df, x='event_domain', y='upf_backlog_bytes_embb', hue='event_domain', legend=False, order=domain_order, ax=axes[1], palette=palette)
axes[1].set_title('Distribusi Kedalaman Antrean Kernel UPF-1', fontweight='bold')
axes[1].set_xlabel('Domain Label')
axes[1].set_ylabel('Backlog Bytes')
axes[1].grid(True, ls='--', alpha=0.5)

plt.suptitle('Distribusi Fitur Fisik Berdasarkan Label Domain', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 2. Tabel statistik deskriptif kuantitatif per domain target
features_eval = ['ran_prb_cell1', 'upf_backlog_bytes_embb', 'nwdaf_comm_ul_bytes']
stats_per_domain = df.groupby('event_domain')[features_eval].agg(['count', 'mean', 'std', 'median', 'min', 'max']).round(2)
print('=== STATISTIK DESKRIPTIF FITUR FISIK PER DOMAIN TARGET ===')
display(stats_per_domain)
